### Start with filtered BMRB

1. Split each entry into different assignment experiment. At the same time, save the criterion related information for further screening.

In [ ]:
import makeshift as ms
import seaborn as sns
import matplotlib.pyplot as plt

import os
import glob
import tqdm
import pickle

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
atom_types = ['H','HA','N','CA','CB', 'C']
result_dict = {}

for name in tqdm.tqdm(os.listdir(data_path)):
    if name.startswith('.'):
        continue

    entry_id = name.split("bmr")[1]
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

        entry = ms.parse_nmr_star(str_files[0])
        cs = ms.get_chem_shifts(entry)

        # Some entries have several cs_saveframe_id, instead of averaging them, we will just note them and process them later
        num_cs = cs['cs_saveframe_id'].unique()
        assign_index = 1
        for assign_name in entry['assigned_chemical_shifts'].keys():
            entry_key = f"{name}_{assign_index}"
            result_dict[entry_key] = {}

            # Get the sequence of the polymer entity in this entry
            entities = ms.get_sequences(entry)
            sequence = entities['Polymer_seq_one_letter_code'].values[0]
            result_dict[entry_key]["sequence"] = sequence

            # Get the sample label of each assignment experiment
            try:
                chem_shift_exp = entry['assigned_chemical_shifts'][assign_name]["_Chem_shift_experiment"]
                all_labels = [item['Sample_label'] for item in chem_shift_exp]
            except KeyError:
                print(f"Chem_shift_experiment is missing for {assign_name} in {name}, skipping.")
                all_labels = []

            " Collect isotopic labeling information "
            sample_info = []
            for sample_name in all_labels:
                # Go with all sample if no sample label is provided
                if sample_name == '.':
                    sample_name = list(entry['sample'].keys())
                else:
                    sample_name = sample_name.split("$")[1]
                try:
                    # Because all the entries are guaranteed to have single entity inside
                    entity_name = list(entry['entity'].keys())[0]
                    # Make sure sample condition and entity we are interested in are consistent
                    if isinstance(sample_name, list):
                        for s in sample_name:
                            sample_info.extend([j['Isotopic_labeling'] for i, j in enumerate(entry['sample'][s]['_Sample_component']) if entity_name in entry['sample'][s]['_Sample_component'][i]['Entity_label']])
                    else:
                        sample_info.extend([j['Isotopic_labeling'] for i, j in enumerate(entry['sample'][sample_name]['_Sample_component']) if entity_name in entry['sample'][sample_name]['_Sample_component'][i]['Entity_label']])
                except KeyError:
                    print(f"Some information is missing for {sample_name} in {name}")
                result_dict[entry_key]['sample_info'] = sample_info 
                
            if not sample_info:
                result_dict[entry_key]['sample_info'] = None

            " Collect isotopic labeling information "
            try:
                reference_tag = entry['assigned_chemical_shifts'][assign_name]['Chem_shift_reference_label']
                # Go with first reference if no reference label is provided, but make sure there's only one reference in this case
                if reference_tag == '.':
                    reference_tag = list(entry['chem_shift_reference'].keys())[0]
                    assert len(entry['chem_shift_reference'].keys()) == 1, f"Multiple chem_shift_reference found for {name} but no reference label provided."
                else:
                    reference_tag = reference_tag.split("$")[1]
                reference_name = entry['chem_shift_reference'][reference_tag]['_Chem_shift_ref']
                reference_info = [item['Mol_common_name'] for item in reference_name]
                result_dict[entry_key]['reference'] = list(set(reference_info))
            except (KeyError, IndexError):
                print(f"Chem_shift_reference is missing for {assign_name} in {name}, skipping.")
                result_dict[entry_key]['reference'] = None

            " Collect experimental conditions "
            condition_name = entry['assigned_chemical_shifts'][assign_name]["Sample_condition_list_label"]
            if condition_name == '.':
                condition_name = list(entry['sample_conditions'].keys())[0]
                assert len(entry['sample_conditions'].keys()) == 1, f"Multiple sample_conditions found for {name} but no condition label provided."
            else:   
                condition_name = condition_name.split("$")[1]
            for condi in entry['sample_conditions'][condition_name]["_Sample_condition_variable"]:
                if condi['Type'].lower() == 'ph':
                    result_dict[entry_key]['pH'] = condi['Val']
                elif condi['Type'].lower() == 'temperature':
                    result_dict[entry_key]['temperature'] = condi['Val']
            if 'pH' not in result_dict[entry_key]:
                result_dict[entry_key]['pH'] = None
            if 'temperature' not in result_dict[entry_key]:
                result_dict[entry_key]['temperature'] = None
    
            result_dict[entry_key]['chemical shifts'] = cs[cs['cs_saveframe_id'] == assign_name]
            assign_index += 1
        
" For downstream analysis "
with open(os.path.join(base_path, "bmrb_cs_data_raw.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)

In [16]:
import pickle
import os

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

print("There are", len(result_dict.keys()), "entries in the dataset.")

There are 11131 entries in the dataset.


In [17]:
from pprint import pprint
pprint(result_dict['bmr52160_1'])

{'chemical shifts':    Entity_ID  Seq_ID Auth_seq_ID Comp_ID Atom_ID Atom_type      Val name  \
0          1       1         566     THR      CA         C   61.806    .   
1          1       1         566     THR      CB         C   69.678    .   
2          1       1         566     THR       N         N  113.302    .   
3          1       2         567     ASN       H         H    8.508    .   
4          1       2         567     ASN      CA         C   53.320    .   
..       ...     ...         ...     ...     ...       ...      ...  ...   
67         1      29         594     ARG      CA         C   55.884    .   
68         1      29         594     ARG      CB         C   30.892    .   
69         1      29         594     ARG       N         N  120.830    .   
70         1      30         595     ALA       H         H    7.948    .   
71         1      30         595     ALA       N         N  110.969    .   

               cs_saveframe_id  
0   assigned_chemical_shifts_1  
1

2. Screen those with deuteration

In [18]:
import pickle
import os
import tqdm

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

deuterate = 0
real_deuterate = 0
real_deuterate_entries = []

for name in tqdm.tqdm(list(result_dict.keys())):
    ifdeuterate = result_dict[name]['sample_info']
    if ifdeuterate is None:
        del result_dict[name]
        deuterate += 1
    elif any("2h" in info.lower() for info in ifdeuterate):
        del result_dict[name]
        deuterate += 1
        real_deuterate += 1
        real_deuterate_entries.append(name)

print(f"Removed {deuterate} entries due to deuteration or unclear info.")
print(f"Among them, {real_deuterate} entries are confirmed to be deuterated.")

with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)
with open(os.path.join(base_path, "removed_entries_due_to_deuterate.txt"), 'w') as f:
    f.write("Entries removed due to deuteration:\n")
    for name in real_deuterate_entries:
        f.write(f"{name}\n")

100%|██████████| 11131/11131 [00:00<00:00, 55967.23it/s]


Removed 1767 entries due to deuteration or unclear info.
Among them, 944 entries are confirmed to be deuterated.


3. Screen those with pH out of range (4.5, 8.5)

In [19]:
import pickle
import os
import tqdm

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

badph = 0
real_badph = 0
real_badph_entries = []

for name in tqdm.tqdm(list(result_dict.keys())):
    ph = result_dict[name]['pH']
    try:
        ph_value = float(ph)
    except (ValueError, TypeError):
        if name in result_dict: del result_dict[name]
        badph += 1
        continue 

    if ph_value < 4.5 or ph_value > 8.5:
        if name in result_dict: del result_dict[name]
        badph += 1
        real_badph += 1
        real_badph_entries.append(name)
print(f"Removed {badph} entries due to pH issues or unclear info.")
print(f"Among them, {real_badph} entries have confirmed pH values outside the acceptable range.")

with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)
with open(os.path.join(base_path, "removed_entries_due_to_ph.txt"), 'w') as f:
    f.write("Entries removed due to pH issues:\n")
    for name in real_badph_entries:
        f.write(f"{name}\n")

100%|██████████| 9364/9364 [00:00<00:00, 201697.07it/s]


Removed 1134 entries due to pH issues or unclear info.
Among them, 899 entries have confirmed pH values outside the acceptable range.


4. Screen those with temperature out of range (293, 313)

In [20]:
import pickle
import os
import tqdm

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

badtemp = 0
real_badtemp = 0
real_badtemp_entries = []

for name in tqdm.tqdm(list(result_dict.keys())):
    temp = result_dict[name]['temperature']
    try:
        temp_value = float(temp)
    except (ValueError, TypeError):
        if name in result_dict: del result_dict[name]
        badtemp += 1
        continue 

    if temp_value < 293 or temp_value > 313:
        if name in result_dict: del result_dict[name]
        badtemp += 1
        real_badtemp += 1
        real_badtemp_entries.append(name)
print(f"Removed {badtemp} entries due to temperature issues or unclear info.")
print(f"Among them, {real_badtemp} entries have confirmed temperature values outside the acceptable range.")

with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph_notemp.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)
with open(os.path.join(base_path, "removed_entries_due_to_temperature.txt"), 'w') as f:
    f.write("Entries removed due to temperature issues:\n")
    for name in real_badtemp_entries:
        f.write(f"{name}\n")

100%|██████████| 8230/8230 [00:00<00:00, 41696.86it/s]


Removed 1374 entries due to temperature issues or unclear info.
Among them, 1357 entries have confirmed temperature values outside the acceptable range.


In [21]:
with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph_notemp.pkl"), 'rb') as f:
    result_dict = pickle.load(f)
print("There are", len(result_dict.keys()), "entries in the dataset.")

There are 6856 entries in the dataset.


5. Re-reference using PANAV method

In [ ]:
import numpy as np
import os
import json
import pickle
import tqdm
import glob
import makeshift as ms

# Initialization for the final dataset
base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
atom_types = ['H','HA','N','CA','CB', 'C']
result_dict = {atom: {} for atom in atom_types}
result_dict.update({atom + "_mask": {} for atom in atom_types})
result_dict["entry_ID"] = {}
result_dict["sequence"] = {}

idx = 1
out_of_bounds = set()
multi_cs = set()
mismatch = set()
notfit_refer = set()

# Helper function to get chemical shift sequence and cs_id mapping
three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

# Main processing loop
with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph_notemp.pkl"), 'rb') as f:
    rawdata = pickle.load(f)

for name in tqdm.tqdm(list(rawdata.keys())):
    folder_name = name.split("_")[0]
    folder_path = os.path.join(data_path, folder_name)
    str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

    key = f"{idx}"

    df = rawdata[name]["chemical shifts"]
    df['Seq_ID'] = df['Seq_ID'].astype(int)
    df = df.sort_values(by='Seq_ID').reset_index(drop=True)

    df_panav, check_panav, offsets_panav = ms.reref(df, method='panav')
    
    if df_panav is None:
        notfit_refer.add(name)
        continue

    # If no re-reference for N, CA, CB, C in this entry
    all_zero = all(value == False for value in check_panav.values())
    if all_zero:
        notfit_refer.add(name)             
        continue
    
    sequence = rawdata[name]["sequence"]
    result_dict["entry_ID"][key] = name
    result_dict["sequence"][key] = sequence

    for atom in atom_types:
        cs_atom = np.full(len(sequence), None, dtype=object)
        cs_atom_mask = np.full(len(sequence), False, dtype=bool)
        sub_df_panav = df_panav[df_panav['Atom_ID'] == atom]
        for _, row in sub_df_panav.iterrows():
            seq_idx_cs = int(row['Seq_ID'])
            outlier_mask_1 = row['outlier_1']
            outlier_mask_2 = row['outlier_2']
            try:
                if three_to_one[row["Comp_ID"]] != sequence[seq_idx_cs - 1].upper():
                    print(f"Residue mismatch at Seq_ID {row['Seq_ID']} in {name}")
                    mismatch.add(name)
                else:
                    cs_atom[seq_idx_cs - 1] = row['Val'] 
                    # For cs_atom_mask, True means it's a valid chemical shift
                    cs_atom_mask[seq_idx_cs - 1] = not outlier_mask_1 and not outlier_mask_2
            except IndexError:
                print(f"Seq_ID {row['Seq_ID']} out of bounds in {name}")     
                out_of_bounds.add(name)

        result_dict[atom][key] = cs_atom.tolist()     
        result_dict[atom + "_mask"][key] = cs_atom_mask.tolist()

    idx += 1

# Remove those where inconsistency happens
mismatch_keys = {k for k, v in result_dict["entry_ID"].items() if v in mismatch}
out_of_bounds_keys = {k for k, v in result_dict["entry_ID"].items() if v in out_of_bounds}
remove_keys = mismatch_keys.union(out_of_bounds_keys)
for field in result_dict.keys():
    for k in remove_keys:
        result_dict[field].pop(k, None)

with open(os.path.join(base_path, "bmrb_cs_data_panav.json"), 'w') as f:
    json.dump(result_dict, f, indent=4)

print(f"Processed {idx-1} entries, results saved to bmrb_cs_data_panav.json")
print(f"There are {len(out_of_bounds)} entries with out-of-bounds Seq_IDs:", out_of_bounds)
print(f"There are {len(multi_cs)} entries with multiple cs_saveframe_id:", multi_cs)
print(f"There are {len(mismatch)} entries with residue mismatches:", mismatch)
print(f"There are {len(notfit_refer)} entries with no HA after rereferencing:", notfit_refer)

6. Re-reference using LACS method

In [ ]:
import numpy as np
import os
import json
import pickle
import tqdm
import glob
import makeshift as ms

# Initialization for the final dataset
base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
atom_types = ['H','N','CA','CB', 'C']
result_dict = {atom: {} for atom in atom_types}
result_dict.update({atom + "_mask": {} for atom in atom_types})
result_dict["entry_ID"] = {}
result_dict["sequence"] = {}

idx = 1
out_of_bounds = set()
multi_cs = set()
mismatch = set()
notfit_refer = set()

# Helper function to get chemical shift sequence and cs_id mapping
three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
    }

# Main processing loop
with open(os.path.join(base_path, "bmrb_cs_data_raw_nodeu_noph_notemp.pkl"), 'rb') as f:
    rawdata = pickle.load(f)

for name in tqdm.tqdm(list(rawdata.keys())):
    folder_name = name.split("_")[0]
    folder_path = os.path.join(data_path, folder_name)
    str_files = glob.glob(os.path.join(folder_path, "*_3.str"))

    key = f"{idx}"

    df = rawdata[name]["chemical shifts"]
    df['Seq_ID'] = df['Seq_ID'].astype(int)
    df = df.sort_values(by='Seq_ID').reset_index(drop=True)

    df_lacs, check_lacs, offsets_lacs = ms.reref(df, method='lacs')
    
    if df_lacs is None:
        notfit_refer.add(name)
        continue

    # If no re-reference for N, CA, CB, C in this entry
    all_zero = all(value == False for value in check_lacs.values())
    if all_zero:
        notfit_refer.add(name)             
        continue
    
    sequence = rawdata[name]["sequence"]
    result_dict["entry_ID"][key] = name
    result_dict["sequence"][key] = sequence

    for atom in atom_types:
        cs_atom = np.full(len(sequence), None, dtype=object)
        cs_atom_mask = np.full(len(sequence), False, dtype=bool)
        sub_df_lacs = df_lacs[df_lacs['Atom_ID'] == atom]
        for _, row in sub_df_lacs.iterrows():
            seq_idx_cs = int(row['Seq_ID'])
            reref_mask = row['reref_mask']
            try:
                if three_to_one[row["Comp_ID"]] != sequence[seq_idx_cs - 1].upper():
                    print(f"Residue mismatch at Seq_ID {row['Seq_ID']} in {name}")
                    mismatch.add(name)
                else:
                    cs_atom[seq_idx_cs - 1] = row['Val'] 
                    # For cs_atom_mask, True means it's a valid chemical shift
                    cs_atom_mask[seq_idx_cs - 1] = reref_mask
            except IndexError:
                print(f"Seq_ID {row['Seq_ID']} out of bounds in {name}")     
                out_of_bounds.add(name)

        result_dict[atom][key] = cs_atom.tolist()     
        result_dict[atom + "_mask"][key] = cs_atom_mask.tolist()

    idx += 1

# Remove those where inconsistency happens
mismatch_keys = {k for k, v in result_dict["entry_ID"].items() if v in mismatch}
out_of_bounds_keys = {k for k, v in result_dict["entry_ID"].items() if v in out_of_bounds}
remove_keys = mismatch_keys.union(out_of_bounds_keys)
for field in result_dict.keys():
    for k in remove_keys:
        result_dict[field].pop(k, None)

with open(os.path.join(base_path, "bmrb_cs_data_lacs.json"), 'w') as f:
    json.dump(result_dict, f, indent=4)

print(f"Processed {idx-1} entries, results saved to bmrb_cs_data_lacs.json")
print(f"There are {len(out_of_bounds)} entries with out-of-bounds Seq_IDs:", out_of_bounds)
print(f"There are {len(multi_cs)} entries with multiple cs_saveframe_id:", multi_cs)
print(f"There are {len(mismatch)} entries with residue mismatches:", mismatch)
print(f"There are {len(notfit_refer)} entries with no HA after rereferencing:", notfit_refer)